<a href="https://colab.research.google.com/github/DenisBoytsov41/tutors-sentiment-coursework/blob/main/notebooks/02_extract_domain_texts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02_extract_domain_texts

Цель ноутбука:
- извлечь полезные текстовые поля из сырых датасетов;
- привести их к единой схеме;
- отдельно сохранить промежуточные CSV по каждому источнику;
- собрать объединённый файл для следующих этапов.

Принцип извлечения:
- **RuSentiment** — базовый русскоязычный корпус тональности;
- **Eedi** — только subset `anchored-dialogues`, потому что он содержит реальные реплики tutor/student;
- **ESConv** — реплики из поддерживающих диалогов;
- **Student Feedback** — тексты образовательных отзывов студентов;
- **project_team_analog** — пока пустой источник.

In [20]:
!pip -q install pandas pyarrow

In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
import json
from pathlib import Path

import pandas as pd

In [23]:
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/tutors_sentiment_project")

RAW_DIR = DRIVE_PROJECT_ROOT / "01_data_raw"
INTERIM_DIR = DRIVE_PROJECT_ROOT / "02_data_interim"

EXTRACTED_DIR = INTERIM_DIR / "extracted_csv"
MERGED_DIR = INTERIM_DIR / "merged"
NORMALIZED_DIR = INTERIM_DIR / "normalized"
DEDUP_DIR = INTERIM_DIR / "deduplicated"

for d in [EXTRACTED_DIR, MERGED_DIR, NORMALIZED_DIR, DEDUP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("RAW_DIR       =", RAW_DIR)
print("EXTRACTED_DIR =", EXTRACTED_DIR)
print("MERGED_DIR    =", MERGED_DIR)

RAW_DIR       = /content/drive/MyDrive/tutors_sentiment_project/01_data_raw
EXTRACTED_DIR = /content/drive/MyDrive/tutors_sentiment_project/02_data_interim/extracted_csv
MERGED_DIR    = /content/drive/MyDrive/tutors_sentiment_project/02_data_interim/merged


In [24]:
UNIFIED_COLUMNS = [
    "dataset_name",
    "source_file",
    "conversation_id",
    "turn_id",
    "speaker_role",
    "text",
    "gold_label",
    "problem_type",
    "emotion_type",
    "situation",
    "split",
    "extra_info",
]

pd.DataFrame({"column": UNIFIED_COLUMNS})

,column
0,dataset_name
1,source_file
2,conversation_id
3,turn_id
4,speaker_role
5,text
6,gold_label
7,problem_type
8,emotion_type
9,situation


In [25]:
def try_read_csv(path: Path):
    for sep in [",", "\t", ";", None]:
        try:
            if sep is None:
                df = pd.read_csv(path, engine="python")
            else:
                df = pd.read_csv(path, sep=sep, engine="python")
            if df.shape[1] >= 1:
                return df
        except Exception:
            continue
    raise ValueError(f"Не удалось прочитать файл: {path}")


def find_first_existing(columns, candidates):
    lower_map = {str(c).lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def safe_text(x):
    if pd.isna(x):
        return None
    x = str(x).replace("\n", " ").strip()
    return x if x else None


def build_unified_df(rows):
    df = pd.DataFrame(rows)
    for col in UNIFIED_COLUMNS:
        if col not in df.columns:
            df[col] = None
    df = df[UNIFIED_COLUMNS].copy()
    df["text"] = df["text"].map(safe_text)
    df = df[df["text"].notna()].copy()
    return df

In [26]:
# извлечение из RuSentiment

rus_dir = RAW_DIR / "rusentiment"
rus_files = sorted(rus_dir.glob("*.csv"))

rus_rows = []

for file_path in rus_files:
    df = try_read_csv(file_path)
    print(file_path.name, df.shape, list(df.columns))

    text_col = find_first_existing(
        df.columns,
        ["text", "content", "message", "post", "sentence", "comment"]
    )
    label_col = find_first_existing(
        df.columns,
        ["label", "labels", "sentiment", "tone", "class"]
    )

    if text_col is None:
        print("  -> текстовый столбец не найден, файл пропущен")
        continue

    for idx, row in df.iterrows():
        rus_rows.append({
            "dataset_name": "rusentiment",
            "source_file": file_path.name,
            "conversation_id": None,
            "turn_id": idx,
            "speaker_role": None,
            "text": row.get(text_col),
            "gold_label": row.get(label_col) if label_col else None,
            "problem_type": None,
            "emotion_type": None,
            "situation": None,
            "split": file_path.stem,
            "extra_info": None,
        })

rus_df = build_unified_df(rus_rows)
rus_df.to_csv(EXTRACTED_DIR / "rusentiment_texts.csv", index=False, encoding="utf-8-sig")

print("rus_df shape:", rus_df.shape)
rus_df.head()

rusentiment_preselected_posts.csv (6950, 2) ['label', 'text']
rusentiment_random_posts.csv (21268, 2) ['label', 'text']
rusentiment_test.csv (2967, 2) ['label', 'text']
rus_df shape: (31185, 12)


,dataset_name,source_file,conversation_id,turn_id,speaker_role,text,gold_label,problem_type,emotion_type,situation,split,extra_info
0,rusentiment,rusentiment_preselected_posts.csv,None,0,None,Прорвём информационную блокаду изнутри.,neutral,None,None,None,rusentiment_preselected_posts,None
1,rusentiment,rusentiment_preselected_posts.csv,None,1,None,"Никогда у меня не будет ""одного приложения для...",negative,None,None,None,rusentiment_preselected_posts,None
2,rusentiment,rusentiment_preselected_posts.csv,None,2,None,"Кури-и тебя не укусит злая собака, потому что ...",skip,None,None,None,rusentiment_preselected_posts,None
3,rusentiment,rusentiment_preselected_posts.csv,None,3,None,"Есть 3 типа людей: Умные, которые делают все с...",neutral,None,None,None,rusentiment_preselected_posts,None
4,rusentiment,rusentiment_preselected_posts.csv,None,4,None,мегафон чет накрыло,neutral,None,None,None,rusentiment_preselected_posts,None


In [27]:
if "gold_label" in rus_df.columns:
    rus_label_summary = (
        rus_df["gold_label"]
        .fillna("<none>")
        .astype(str)
        .value_counts(dropna=False)
        .reset_index()
    )
    rus_label_summary.columns = ["gold_label", "count"]
else:
    rus_label_summary = pd.DataFrame(columns=["gold_label", "count"])

rus_label_summary.to_csv(EXTRACTED_DIR / "rusentiment_label_summary.csv", index=False, encoding="utf-8-sig")
rus_label_summary

,gold_label,count
0,neutral,12720
1,positive,6646
2,skip,4440
3,negative,3912
4,speech,3467


In [28]:
# извлечение из Eedi: только anchored-dialogues

eedi_dir = RAW_DIR / "eedi_tutoring"
anchored_dir = eedi_dir / "anchored-dialogues"

eedi_rows = []

eedi_parquet_files = sorted(anchored_dir.glob("*.parquet"))
eedi_csv_files = sorted(anchored_dir.glob("*.csv"))

print("anchored-dialogues parquet:", len(eedi_parquet_files))
print("anchored-dialogues csv    :", len(eedi_csv_files))

anchored-dialogues parquet: 2
anchored-dialogues csv    : 2


In [29]:
candidate_files = eedi_parquet_files + eedi_csv_files

for file_path in candidate_files:
    try:
        if file_path.suffix.lower() == ".parquet":
            df = pd.read_parquet(file_path)
        else:
            df = try_read_csv(file_path)
    except Exception as e:
        print("Ошибка чтения:", file_path, e)
        continue

    print(file_path.name, df.shape, list(df.columns))

    text_col = find_first_existing(
        df.columns,
        ["MessageString", "message", "text", "content", "utterance", "sentence"]
    )
    role_col = find_first_existing(
        df.columns,
        ["IsTutor", "speaker", "role", "speaker_role"]
    )
    conv_col = find_first_existing(
        df.columns,
        ["InterventionId", "conversation_id", "dialogue_id", "dialog_id"]
    )
    turn_col = find_first_existing(
        df.columns,
        ["MessageSequence", "turn_id", "turn_index", "TurnIndex"]
    )
    talk_move_col = find_first_existing(
        df.columns,
        ["TalkMovePrediction", "talk_move_prediction"]
    )

    if text_col is None:
        print("  -> текстовый столбец не найден, файл пропущен")
        continue

    for idx, row in df.iterrows():
        role_value = row.get(role_col) if role_col else None

        if role_col == "IsTutor":
            speaker_role = "tutor" if str(role_value).lower() in ["1", "true", "yes"] else "student"
        else:
            speaker_role = role_value

        extra_payload = {}
        if talk_move_col:
            extra_payload["TalkMovePrediction"] = row.get(talk_move_col)

        eedi_rows.append({
            "dataset_name": "eedi_tutoring",
            "source_file": str(file_path.relative_to(eedi_dir)),
            "conversation_id": row.get(conv_col) if conv_col else None,
            "turn_id": row.get(turn_col) if turn_col else idx,
            "speaker_role": speaker_role,
            "text": row.get(text_col),
            "gold_label": None,
            "problem_type": None,
            "emotion_type": None,
            "situation": None,
            "split": file_path.stem,
            "extra_info": json.dumps(extra_payload, ensure_ascii=False) if extra_payload else None,
        })

eedi_df = build_unified_df(eedi_rows)
eedi_df.to_csv(EXTRACTED_DIR / "eedi_texts.csv", index=False, encoding="utf-8-sig")

print("eedi_df shape:", eedi_df.shape)
eedi_df.head()

test-00000-of-00001.parquet (13395, 7) ['InterventionId', 'UserId', 'QuestionId_DQ', 'MessageSequence', 'IsTutor', 'MessageString', 'TalkMovePrediction']
train-00000-of-00001.parquet (55322, 7) ['InterventionId', 'UserId', 'QuestionId_DQ', 'MessageSequence', 'IsTutor', 'MessageString', 'TalkMovePrediction']
test.csv (13395, 7) ['InterventionId', 'TutorId', 'QuestionId_DQ', 'MessageSequence', 'IsTutor', 'MessageString', 'TalkMovePrediction']
train.csv (55322, 7) ['InterventionId', 'TutorId', 'QuestionId_DQ', 'MessageSequence', 'IsTutor', 'MessageString', 'TalkMovePrediction']
eedi_df shape: (137425, 12)


,dataset_name,source_file,conversation_id,turn_id,speaker_role,text,gold_label,problem_type,emotion_type,situation,split,extra_info
0,eedi_tutoring,anchored-dialogues/test-00000-of-00001.parquet,172,1,tutor,hello how are you?,None,None,None,None,test-00000-of-00001,"{""TalkMovePrediction"": ""<None>""}"
1,eedi_tutoring,anchored-dialogues/test-00000-of-00001.parquet,172,2,student,fine,None,None,None,None,test-00000-of-00001,"{""TalkMovePrediction"": null}"
2,eedi_tutoring,anchored-dialogues/test-00000-of-00001.parquet,172,3,tutor,great!,None,None,None,None,test-00000-of-00001,"{""TalkMovePrediction"": ""<None>""}"
3,eedi_tutoring,anchored-dialogues/test-00000-of-00001.parquet,172,4,tutor,Would you like me to look at this question wit...,None,None,None,None,test-00000-of-00001,"{""TalkMovePrediction"": ""<Keep Together>""}"
4,eedi_tutoring,anchored-dialogues/test-00000-of-00001.parquet,172,5,student,yes,None,None,None,None,test-00000-of-00001,"{""TalkMovePrediction"": null}"


In [30]:
# сводка по ролям Eedi

eedi_role_summary = (
    eedi_df["speaker_role"]
    .fillna("<none>")
    .astype(str)
    .value_counts(dropna=False)
    .reset_index()
)
eedi_role_summary.columns = ["speaker_role", "count"]

eedi_role_summary.to_csv(EXTRACTED_DIR / "eedi_role_summary.csv", index=False, encoding="utf-8-sig")
eedi_role_summary

,speaker_role,count
0,tutor,81612
1,student,55813


In [31]:
#извлечение из ESConv

esconv_path = RAW_DIR / "esconv" / "ESConv.json"

esconv_rows = []

if esconv_path.exists():
    with open(esconv_path, "r", encoding="utf-8") as f:
        esconv_data = json.load(f)

    for conv_idx, conv in enumerate(esconv_data):
        dialog = conv.get("dialog", [])
        for turn_idx, turn in enumerate(dialog):
            annotation = turn.get("annotation", {}) or {}

            esconv_rows.append({
                "dataset_name": "esconv",
                "source_file": "ESConv.json",
                "conversation_id": conv_idx,
                "turn_id": turn_idx,
                "speaker_role": turn.get("speaker"),
                "text": turn.get("content"),
                "gold_label": None,
                "problem_type": conv.get("problem_type"),
                "emotion_type": conv.get("emotion_type"),
                "situation": conv.get("situation"),
                "split": None,
                "extra_info": json.dumps(annotation, ensure_ascii=False) if annotation else None,
            })

esconv_df = build_unified_df(esconv_rows)
esconv_df.to_csv(EXTRACTED_DIR / "esconv_texts.csv", index=False, encoding="utf-8-sig")

print("esconv_df shape:", esconv_df.shape)
esconv_df.head()

esconv_df shape: (38365, 12)


,dataset_name,source_file,conversation_id,turn_id,speaker_role,text,gold_label,problem_type,emotion_type,situation,split,extra_info
0,esconv,ESConv.json,0,0,seeker,Hello,None,job crisis,anxiety,I hate my job but I am scared to quit and seek...,None,None
1,esconv,ESConv.json,0,1,supporter,"Hello, what would you like to talk about?",None,job crisis,anxiety,I hate my job but I am scared to quit and seek...,None,"{""strategy"": ""Question""}"
2,esconv,ESConv.json,0,2,seeker,I am having a lot of anxiety about quitting my...,None,job crisis,anxiety,I hate my job but I am scared to quit and seek...,None,None
3,esconv,ESConv.json,0,3,supporter,What makes your job stressful for you?,None,job crisis,anxiety,I hate my job but I am scared to quit and seek...,None,"{""strategy"": ""Question""}"
4,esconv,ESConv.json,0,4,seeker,I have to deal with many people in hard financ...,None,job crisis,anxiety,I hate my job but I am scared to quit and seek...,None,"{""feedback"": ""5""}"


In [32]:
# сводка по ролям ESConv

esconv_role_summary = (
    esconv_df["speaker_role"]
    .fillna("<none>")
    .astype(str)
    .value_counts(dropna=False)
    .reset_index()
)
esconv_role_summary.columns = ["speaker_role", "count"]

esconv_role_summary.to_csv(EXTRACTED_DIR / "esconv_role_summary.csv", index=False, encoding="utf-8-sig")
esconv_role_summary

,speaker_role,count
0,seeker,19989
1,supporter,18376


In [33]:
# извлечение из Student Feedback

student_dir = RAW_DIR / "student_feedback"
student_tsv_files = sorted(student_dir.rglob("*.tsv"))

student_rows = []

print("TSV files:", len(student_tsv_files))
for p in student_tsv_files[:20]:
    print(" ", p)

TSV files: 5
  /content/drive/MyDrive/tutors_sentiment_project/01_data_raw/student_feedback/Annotated Student Feedback Data/Annotator_1/towe-eacl_recreation_data_set/defomative comment removed/train.tsv
  /content/drive/MyDrive/tutors_sentiment_project/01_data_raw/student_feedback/Annotated Student Feedback Data/Annotator_1/towe-eacl_recreation_data_set/less than 100 lengthy comment/test.tsv
  /content/drive/MyDrive/tutors_sentiment_project/01_data_raw/student_feedback/Annotated Student Feedback Data/Annotator_1/towe-eacl_recreation_data_set/less than 100 lengthy comment/train.tsv
  /content/drive/MyDrive/tutors_sentiment_project/01_data_raw/student_feedback/Annotated Student Feedback Data/Annotator_1/towe-eacl_recreation_data_set/test.tsv
  /content/drive/MyDrive/tutors_sentiment_project/01_data_raw/student_feedback/Annotated Student Feedback Data/Annotator_1/towe-eacl_recreation_data_set/train.tsv


In [34]:
# чтение Student Feedback

for file_path in student_tsv_files:
    try:
        df = pd.read_table(file_path, header=None, encoding="utf-8")
    except Exception as e:
        print("Ошибка чтения:", file_path, e)
        continue

    if df.shape[1] == 4:
        df.columns = ["id", "text", "tags_1", "tags_2"]
        text_col = "text"
    else:
        text_col = find_first_existing(
            df.columns,
            ["text", "content", "message", "comment", "sentence"]
        )

    if text_col is None:
        continue

    for idx, row in df.iterrows():
        student_rows.append({
            "dataset_name": "student_feedback",
            "source_file": str(file_path.relative_to(student_dir)),
            "conversation_id": None,
            "turn_id": row.get("id") if "id" in df.columns else idx,
            "speaker_role": "student",
            "text": row.get(text_col),
            "gold_label": None,
            "problem_type": None,
            "emotion_type": None,
            "situation": None,
            "split": file_path.stem,
            "extra_info": None,
        })

student_df = build_unified_df(student_rows)
student_df.to_csv(EXTRACTED_DIR / "student_feedback_texts.csv", index=False, encoding="utf-8-sig")

print("student_df shape:", student_df.shape)
student_df.head()

student_df shape: (16631, 12)


,dataset_name,source_file,conversation_id,turn_id,speaker_role,text,gold_label,problem_type,emotion_type,situation,split,extra_info
0,student_feedback,Annotated Student Feedback Data/Annotator_1/to...,None,11,student,Throughout the whole semester our focus was on...,None,None,None,None,train,None
1,student_feedback,Annotated Student Feedback Data/Annotator_1/to...,None,20,student,Duration for the OOSD semester project should ...,None,None,None,None,train,None
2,student_feedback,Annotated Student Feedback Data/Annotator_1/to...,None,31,student,Duration for the OOSD semester project should ...,None,None,None,None,train,None
3,student_feedback,Annotated Student Feedback Data/Annotator_1/to...,None,40,student,Both lecturers did a good job on delivering th...,None,None,None,None,train,None
4,student_feedback,Annotated Student Feedback Data/Annotator_1/to...,None,51,student,Both lecturers did a good job on delivering th...,None,None,None,None,train,None


In [35]:
project_team_rows = []

project_team_df = build_unified_df(project_team_rows)
project_team_df.to_csv(EXTRACTED_DIR / "project_team_analog_texts.csv", index=False, encoding="utf-8-sig")

print("project_team_df shape:", project_team_df.shape)
project_team_df.head()

project_team_df shape: (0, 12)


,dataset_name,source_file,conversation_id,turn_id,speaker_role,text,gold_label,problem_type,emotion_type,situation,split,extra_info


In [36]:
# объединение всех источников

all_parts = [rus_df, eedi_df, esconv_df, student_df, project_team_df]
domain_texts_df = pd.concat(all_parts, ignore_index=True)

domain_texts_df = domain_texts_df.drop_duplicates(
    subset=["dataset_name", "speaker_role", "text"]
).reset_index(drop=True)

domain_texts_df.to_csv(MERGED_DIR / "domain_texts_unified.csv", index=False, encoding="utf-8-sig")

print("domain_texts_df shape:", domain_texts_df.shape)
domain_texts_df.head()

domain_texts_df shape: (120189, 12)


,dataset_name,source_file,conversation_id,turn_id,speaker_role,text,gold_label,problem_type,emotion_type,situation,split,extra_info
0,rusentiment,rusentiment_preselected_posts.csv,None,0,None,Прорвём информационную блокаду изнутри.,neutral,None,None,None,rusentiment_preselected_posts,None
1,rusentiment,rusentiment_preselected_posts.csv,None,1,None,"Никогда у меня не будет ""одного приложения для...",negative,None,None,None,rusentiment_preselected_posts,None
2,rusentiment,rusentiment_preselected_posts.csv,None,2,None,"Кури-и тебя не укусит злая собака, потому что ...",skip,None,None,None,rusentiment_preselected_posts,None
3,rusentiment,rusentiment_preselected_posts.csv,None,3,None,"Есть 3 типа людей: Умные, которые делают все с...",neutral,None,None,None,rusentiment_preselected_posts,None
4,rusentiment,rusentiment_preselected_posts.csv,None,4,None,мегафон чет накрыло,neutral,None,None,None,rusentiment_preselected_posts,None


In [37]:
# сводка по источникам

summary_df = (
    domain_texts_df
    .groupby("dataset_name")
    .agg(
        rows=("text", "count"),
        unique_texts=("text", pd.Series.nunique)
    )
    .reset_index()
)

summary_df.to_csv(MERGED_DIR / "domain_texts_summary.csv", index=False, encoding="utf-8-sig")
summary_df

,dataset_name,rows,unique_texts
0,eedi_tutoring,50267,50022
1,esconv,36238,36074
2,rusentiment,31066,31066
3,student_feedback,2618,2618


In [38]:
# сводка по ролям

role_summary_df = (
    domain_texts_df
    .fillna({"speaker_role": "<none>"})
    .groupby(["dataset_name", "speaker_role"])
    .size()
    .reset_index(name="count")
)

role_summary_df.to_csv(MERGED_DIR / "domain_texts_role_summary.csv", index=False, encoding="utf-8-sig")
role_summary_df

,dataset_name,speaker_role,count
0,eedi_tutoring,student,16179
1,eedi_tutoring,tutor,34088
2,esconv,seeker,18895
3,esconv,supporter,17343
4,rusentiment,<none>,31066
5,student_feedback,student,2618


In [39]:
print("=" * 70)
print("Извлечение доменных текстов завершено")
print("=" * 70)
print("Файлы сохранены в:", EXTRACTED_DIR)
print("Объединённый файл :", MERGED_DIR / "domain_texts_unified.csv")

Извлечение доменных текстов завершено
Файлы сохранены в: /content/drive/MyDrive/tutors_sentiment_project/02_data_interim/extracted_csv
Объединённый файл : /content/drive/MyDrive/tutors_sentiment_project/02_data_interim/merged/domain_texts_unified.csv
